In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from ase import units

def ry_au_to_ev_angstrom(force):
    """Convert force(s) from Ry/au to eV/Å
    
    Can handle single float or list/array of values
    """
    if isinstance(force, (list, tuple)):
        return [f * units.Ry / units.Bohr for f in force]
    return force * units.Ry / units.Bohr

In [ ]:
df_md = pd.read_csv("sample_lammps_df.csv", delimiter=" ")
df_qe = pd.read_csv("sample_qe_df.csv", delimiter=" ")

In [ ]:
df_md

In [ ]:
df_qe

In [ ]:
print("Confirm indexing for Quantum ESPRESSO dataframe:\n")

! ls PWSCF-222-100K/*.out -1

df_qe["frame_index"].unique()

In [ ]:
print("Number of frames of MD:", df_md["frame_index"].nunique())
print("Number of frames of DFT:", df_qe["frame_index"].nunique())
print()
print(f"Rows MD = {df_md.shape[0]} = {51*32}")
print(f"Rows DFT = {df_qe.shape[0]} = {5*32}")

In [ ]:
from data_selection import force_parity, force_metrics

In [ ]:
df_parity = force_parity(df_LAMMPS=df_md, df_QE=df_qe, write=False)
force_metrics(df_parity=df_parity, force_error_threshold=13)

In [ ]:
display(df_parity.query("frame_index == 0"))

In [ ]:
display(df_parity.query("frame_index == 0").query("atom_index == 1"))


! echo  'LAMMPS: id type x y z fx fy fz vx vy vz'
! echo
! head -n41  "../thermal_expansion/100K/trajectories/ZnO-222-100K.lammpstrj" | grep -e '^1' | head -n1
! head -n41  "../thermal_expansion/100K/trajectories/ZnO-222-100K.lammpstrj" | grep -e '^1' | head -n1 | awk '{print "Fz = " $8 " eV/A"}'
! echo


! echo 'Quantum ESPRESSO'
! grep -A3 "Forces acting" "./PWSCF-222-100K/frame_pwscf_000.out"
print()
print(f"Fz =  0.00001616 Ry/au =  {ry_au_to_ev_angstrom(0.00001616):.6f} eV/Angstrom")

In [ ]:
df_parity

<h1 align = "center">  Metrics </h1>


## 1. Atom-wise force error

Suppose atom $\xi$ has force vectors $\mathbf{F}_a^{DFT}$ and $\mathbf{F}_a^{MLFF}$, we define the $\color{orange}{\text{force error}}$ as:


$$
e_a = \left\| \mathbf{F}_a^{DFT} - \mathbf{F}_a^{MLFF} \right\| = 
\sqrt{
(F_x^{DFT}-F_x^{MLFF})^2 +
(F_y^{DFT}-F_y^{MLFF})^2 +
(F_z^{DFT}-F_z^{MLFF})^2
}

$$ 

## 2. Frame-level metrics

Upon the *atom-wise* metric we derive the following *frame-level* metrics:

- $\Delta F_{i}^{\max}= \max_{a} e_a \hspace{11mm}$ *Did something Catastrophic happened?*
- $\Delta F_{i}^{RMS} = \sqrt{\frac{1}{N}\sum_{a}e_{a}^{2}}  \hspace{5mm}$ *How bad is the frame overall?*